## 🔗 Follow Me

<p align="center">
  <a href="https://github.com/Ramakm">
    <img src="https://img.shields.io/badge/GitHub-Follow-black?style=for-the-badge&logo=github" />
  </a>
  &nbsp;
  <a href="https://www.instagram.com/techwith.ram/">
    <img src="https://img.shields.io/badge/Instagram-Follow-E4405F?style=for-the-badge&logo=instagram&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/techwith_ram">
    <img src="https://img.shields.io/badge/Twitter-Follow-1DA1F2?style=for-the-badge&logo=twitter&logoColor=white" />
  </a>
</p>

# 04 · Data preparation

## Where to get data
| Dataset | Link | Size | Good for |
|---|---|---|---|
| **FineTome-100k** (default) | https://huggingface.co/datasets/mlabonne/FineTome-100k | 100k | general high-quality instruction following |
| Bitext customer support | https://huggingface.co/datasets/bitext/Bitext-Gen-AI-Chatbot-Customer-Support-Dataset | 27k | customer-support bot |
| Alpaca-cleaned | https://huggingface.co/datasets/yahma/alpaca-cleaned | 52k | simple instruction→response |
| Dolly-15k | https://huggingface.co/datasets/databricks/databricks-dolly-15k | 15k | commercially-usable Q&A / summarisation |
| OpenAssistant OASST2 | https://huggingface.co/datasets/OpenAssistant/oasst2 | 135k | human multi-turn chats |
| Your own data | CSV/JSONL of prompt+answer | 500+ | the best option |

No account is needed for these (they download automatically via 🤗 `datasets`). Change `DATASET` below to switch.

This notebook builds `data/train.jsonl`, `data/val.jsonl`, `data/test.jsonl`.

In [ ]:
!pip install -q datasets transformers pandas

In [ ]:
import json, random, os
from datasets import load_dataset

DATASET   = "mlabonne/FineTome-100k"   # ShareGPT-style: conversations=[{from,value}]
N_SAMPLES = 6000                       # 5-10k is plenty for a style/behaviour fine-tune on a T4
MAX_TOKENS = 1024                      # drop very long examples (memory)
SEED = 42
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

raw = load_dataset(DATASET, split="train")
print(raw)
print(raw[0])

## Convert to the `messages` schema

In [ ]:
ROLE = {"human": "user", "user": "user", "gpt": "assistant", "assistant": "assistant", "system": "system"}

def to_messages(ex):
    msgs = [{"role": ROLE[t["from"]], "content": t["value"].strip()} for t in ex["conversations"]]
    return {"messages": msgs}

ds = raw.shuffle(seed=SEED).select(range(min(len(raw), N_SAMPLES * 2)))   # oversample, filter next
ds = ds.map(to_messages, remove_columns=ds.column_names)

def valid(ex):
    m = ex["messages"]
    roles_ok = m[-1]["role"] == "assistant" and all(
        a["role"] != b["role"] for a, b in zip(m, m[1:]))          # alternating turns, ends with assistant
    return roles_ok and all(len(x["content"]) > 0 for x in m)

ds = ds.filter(valid)
print("after validity filter:", len(ds))

## Filter by token length and de-duplicate

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")

def n_tokens(ex):
    return {"n_tok": len(tok.apply_chat_template(ex["messages"], tokenize=True))}

ds = ds.map(n_tokens)
ds = ds.filter(lambda e: 8 <= e["n_tok"] <= MAX_TOKENS)

seen, keep = set(), []
for i, ex in enumerate(ds):
    key = ex["messages"][0]["content"][:200]
    if key not in seen:
        seen.add(key); keep.append(i)
ds = ds.select(keep).select(range(min(len(keep), N_SAMPLES)))
print("final size:", len(ds))

import pandas as pd
print(pd.Series(ds["n_tok"]).describe())

## Split & save (80 / 10 / 10)

In [ ]:
ds = ds.shuffle(seed=SEED)
n = len(ds); a, b = int(.8*n), int(.9*n)
splits = {"train": ds.select(range(0, a)), "val": ds.select(range(a, b)), "test": ds.select(range(b, n))}

for name, d in splits.items():
    with open(f"{DATA_DIR}/{name}.jsonl", "w") as f:
        for ex in d:
            f.write(json.dumps({"messages": ex["messages"]}, ensure_ascii=False) + "\n")
    print(name, len(d))

print(json.dumps(splits["train"][0]["messages"], indent=2)[:800])

## Using your own data instead
Create a CSV with `instruction,response` columns, then:
```python
import pandas as pd
df = pd.read_csv("my_data.csv")
rows = [{"messages":[{"role":"user","content":r.instruction},{"role":"assistant","content":r.response}]} for r in df.itertuples()]
```
and write `rows` to `data/train.jsonl` / `val.jsonl` / `test.jsonl`.

### Data quality checklist
- Quality ≫ quantity (1k clean examples beat 50k noisy ones)
- Consistent style in the responses you want to imitate
- No test-set leakage (we split *before* training)
- No personal data / secrets
- Check each dataset's license before commercial use

Next → `05_Training.ipynb`